# Vercel 배포 폴더 만들기
- 실행 위치: `app_integrated_v1.py` 가 있는 html 폴더
- 결과: 상위 폴더의 `vercel_hub` (원본은 그대로)

## 1. 설정

In [ ]:
# html 폴더(app_integrated_v1.py 가 있는 곳)에서 실행
# 결과: 상위 폴더에 vercel_hub 폴더 생성 (원본 파일은 건드리지 않음)
import os
import glob
import json
import shutil

SRC = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
ROOTS = [SRC, os.path.dirname(SRC)]                  # html 폴더, 그 상위 폴더
OUT = os.path.join(os.path.dirname(SRC), "vercel_hub")

print("원본 폴더:", SRC)
print("배포 폴더:", OUT)

## 2. 배포 폴더 비우기 (.vercel 연결 정보는 유지)

In [ ]:
os.makedirs(OUT, exist_ok=True)
for name in os.listdir(OUT):
    if name == ".vercel":
        continue
    path = os.path.join(OUT, name)
    if os.path.isdir(path):
        shutil.rmtree(path)
    else:
        os.remove(path)

copied = []

def copy(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    copied.append(os.path.relpath(dst, OUT))

## 3. 서버 파일

In [ ]:
app_file = os.path.join(SRC, "app_integrated_v1.py")
if not os.path.isfile(app_file):
    raise FileNotFoundError("app_integrated_v1.py 가 없습니다. html 폴더에서 실행하세요.")
copy(app_file, os.path.join(OUT, "app_integrated_v1.py"))

## 4. HTML → templates/ (templates 폴더 우선)

In [ ]:
html_files = {}
for root in ROOTS:
    for sub in ("templates", ""):
        for path in glob.glob(os.path.join(root, sub, "*.html")):
            html_files.setdefault(os.path.basename(path), path)

for name, path in html_files.items():
    copy(path, os.path.join(OUT, "templates", name))

## 5. CSS·JS → public/static/ (Vercel CDN 이 /static/... 주소로 제공)

In [ ]:
for root in ROOTS:
    static_dir = os.path.join(root, "static")
    if os.path.isdir(static_dir):
        for path in glob.glob(os.path.join(static_dir, "**", "*"), recursive=True):
            if os.path.isfile(path):
                rel = os.path.relpath(path, static_dir)
                copy(path, os.path.join(OUT, "public", "static", rel))

# static 폴더 밖에 따로 있는 css·js 도 정리
for root in ROOTS:
    for sub in ("templates", ""):
        for ext in ("css", "js"):
            for path in glob.glob(os.path.join(root, sub, f"*.{ext}")):
                dst = os.path.join(OUT, "public", "static", ext, os.path.basename(path))
                if not os.path.exists(dst):
                    copy(path, dst)

## 6. 데이터 파일 (예측 결과, 병상 캐시)

In [ ]:
def find_first(name, subs=("outputs", "static", "data", "")):
    for root in ROOTS:
        for sub in subs:
            path = os.path.join(root, sub, name)
            if os.path.isfile(path):
                return path
    return None

for name in ("dashboard_data.json", "dashboard_data.js"):
    path = find_first(name)
    if path:
        copy(path, os.path.join(OUT, "outputs", name))

cache = find_first("realtime_beds_cache.json", subs=("",))
if cache:
    copy(cache, os.path.join(OUT, "realtime_beds_cache.json"))

## 7. Vercel 설정 파일

In [ ]:
files = {
    "app.py": "# Vercel 진입 파일: app_integrated_v1.py 의 app 을 그대로 사용\n"
              "from app_integrated_v1 import app\n",
    "requirements.txt": "flask\nrequests\npython-dotenv\n",
    "vercel.json": json.dumps({"functions": {"app.py": {"maxDuration": 120}}}, indent=2) + "\n",
    ".vercelignore": ".env\n.venv\n__pycache__\n*.ipynb\n*.bat\nchatbot_exaone.py\n",
}
for name, text in files.items():
    with open(os.path.join(OUT, name), "w", encoding="utf-8") as f:
        f.write(text)
    copied.append(name)

## 8. 결과 확인

In [ ]:
print(f"\n복사·생성한 파일 {len(copied)}개")
for name in sorted(copied):
    print("  ", name)

need = {
    "templates/base.html": "공통 틀",
    "templates/header.html": "머리말",
    "templates/footer.html": "꼬리말",
    "outputs/dashboard_data.json": "예측 결과 (없으면 분석 화면에 안내 문구)",
    "realtime_beds_cache.json": "첫 화면 병상 (없으면 첫 요청에서 수집)",
}
print("\n필수·권장 파일 확인")
for rel, desc in need.items():
    ok = os.path.isfile(os.path.join(OUT, rel))
    print(f"  {'OK  ' if ok else '없음'} {rel}  - {desc}")

home = [n for n in ("main.html", "SEOUL_EMERGENCY_HUB.html") if n in html_files]
print(f"  {'OK  ' if home else '없음'} 메인 화면 html {home}")